## Initialize and create routes

In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

In [ ]:
from pathlib import Path

DATA_RAW = Path("/MUON-SPECTRUM-CNF/data/raw")              # Adjust paths as needed
DATA_PROCESSED = Path("/MUON-SPECTRUM-CNF/data/processed")  # Adjust paths as needed
WORK_BASE = Path("/MUON-SPECTRUM-CNF/results")              # Adjust paths as needed

# Input/output paths
input_dir = DATA_RAW / "shw"
csv_dir   = DATA_PROCESSED
info_file = "/MUON-SPECTRUM-CNF/data/lago_sites_from_jsonld.csv"

out_csv = WORK_BASE / "muon_flux.csv"
OUT_FLUX_FIG = WORK_BASE / "muon_flux_vs_height.png"

# Create output folders
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
WORK_BASE.mkdir(parents=True, exist_ok=True)


if not info_file.exists():
    print("\nMissing file:", info_file.name)
    print("Place it at:", str(info_file))

if len(list(input_dir.glob("*.shw"))) == 0 and len(list(input_dir.glob("*.SHW"))) == 0:
    print("\nNo .shw files found in:", str(input_dir))
    print("Place your .shw files inside that folder.")

In [ ]:
#all simulations correspond to 3600 s
divisor   = 3600.0

# Muon mass in GeV
muon_mass = 0.10566

# The key column is 'sigla', used to match each file/city name
info = pd.read_csv(info_file)
info["sigla"] = info["sigla"].astype(str).str.strip().str.lower()

# Convert decimal comma to decimal point for numeric columns
num_cols = ["lat", "lon", "h", "bi", "bx", "bz"]
for c in num_cols:
    if c in info.columns:
        info[c] = pd.to_numeric(
            info[c].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )

## SHW → CSV preprocessing (muons only)

This cell reads each `.shw` file, filters muons (`CorsikaId` 5 and 6), computes:
- **Muon energy** from `(px, py, pz)` plus the muon mass
- **Zenith angle** `theta` from the momentum direction

In [ ]:
shw_files = glob.glob(os.path.join(input_dir, "*.shw")) + glob.glob(os.path.join(input_dir, "*.SHW"))

for file in shw_files:
    # Standardize filename and enforce "_3600" suffix
    filename = os.path.splitext(os.path.basename(file))[0].strip().lower()
    if not filename.endswith("_3600"):
        filename = filename + "_3600"

    # SHW files are whitespace-separated tables
    df = pd.read_csv(
        file,
        delim_whitespace=True,
        comment="#",
        header=None,
        encoding="latin1"
    )

    # Assign column names expected from the SHW structure
    df.columns = [
        "CorsikaId", "px", "py", "pz",
        "x", "y", "z",
        "shower_id", "prm_id", "prm_energy",
        "prm_theta", "prm_phi"
    ]

    # Keep only muons (Corsika IDs 5 and 6)
    df = df[df["CorsikaId"].isin([5, 6])]

    # Compute momentum magnitude and derived quantities
    p = np.sqrt(df["px"]**2 + df["py"]**2 + df["pz"]**2)
    df["energy"] = np.sqrt(p**2 + muon_mass**2)
    df["theta"]  = np.degrees(np.arccos(df["pz"] / p))

    # Keep only variables used downstream
    df = df[["energy", "theta"]]

    # Attach geophysical/site parameters by matching sigla == filename
    row = info[info["sigla"] == filename]
    city_data = row.drop(columns=["sigla"]).iloc[0].to_dict()
    for key, value in city_data.items():
        df[key] = value

    # Save processed data for this city
    df.to_csv(os.path.join(csv_dir, f"{filename}.csv"), index=False)

## Build `muon_flux.csv` (one-row summary per city)

This cell scans the per-city CSV files and builds a compact summary table:
- `N`: number of muons (rows)
- `flux = N/3600`: average rate assuming a 3600 s window
- plus site metadata (`lat`, `lon`, `h`, `bx`, `bz`, `bi`)

In [ ]:
# These files are the inputs for summary + spectra plots.
files = sorted(glob.glob(os.path.join(csv_dir, "*.csv")))
labels = [os.path.basename(file).split("_")[0] for file in files]

# Each row contains: site metadata + N muons + flux = N / 3600.
rows = []
for file, label in zip(files, labels):
    df = pd.read_csv(file)

    lat = df["lat"].iloc[0]
    lon = df["lon"].iloc[0]
    h   = df["h"].iloc[0]
    bi  = df["bi"].iloc[0]
    bx  = df["bx"].iloc[0]
    bz  = df["bz"].iloc[0]

    n = len(df)
    flux = n / divisor

    rows.append({
        "sigla": label, "lat": lat, "lon": lon, "h": h,
        "N": n, "flux": flux, "bx": bx, "bz": bz, "bi": bi
    })

df_flux = pd.DataFrame(rows)
df_flux["h"] = pd.to_numeric(df_flux["h"], errors="coerce")
df_flux = df_flux.sort_values("h").reset_index(drop=True)
df_flux.to_csv(out_csv, index=False)

## Poisson MLE fit: muon flux vs height

This cell fits a Poisson count model:

$$
N_i \sim \text{Poisson}(\mu_i), \quad \mu_i = T \cdot \exp(a + bH_i), \quad T=3600
$$

It estimates $(a, b)$ by maximum likelihood (Newton-Raphson), computes predicted flux, prints basic metrics (RMSE, $R^2$), and saves a plot.  
**Why:** counts over a fixed time window are naturally modeled with Poisson; the exponential trend provides a simple global relationship between height and muon rate.


In [ ]:
# Global plot style
plt.rcParams.update({
    "figure.figsize": (8, 6),
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 14,
    "lines.linewidth": 2.0
})

In [ ]:
# Model: N_i ~ Poisson(mu_i), where mu_i = T * exp(a + b * H_i), with T=3600.
df = pd.read_csv(out_csv)

L = df["sigla"].astype(str).tolist()
H = df["h"].to_numpy(dtype=float)
Y = df["flux"].to_numpy(dtype=float)
N = df["N"].to_numpy(dtype=float)
T = divisor

def fit_poisson_mle(H, N, T, a0=0.0, b0=0.0, iters=100, tol=1e-10):
    # Newton-Raphson optimization for Poisson log-likelihood
    # Parameters: a (intercept), b (slope), with mu = T * exp(a + bH).
    a, b = a0, b0
    for _ in range(iters):
        mu = T * np.exp(a + b*H)     # expected counts
        r  = mu - N                  # residuals in count space

        # Gradient of the negative log-likelihood
        g1 = np.sum(r)
        g2 = np.sum(r * H)

        # Fisher information (Hessian approximation) for stability
        H11 = np.sum(mu)
        H12 = np.sum(mu * H)
        H22 = np.sum(mu * H * H)
        det = H11*H22 - H12*H12
        if det <= 0:
            break

        # Newton step
        da = -( H22*g1 - H12*g2)/det
        db = -(-H12*g1 + H11*g2)/det
        a += da
        b += db

        if abs(da) + abs(db) < tol:
            break

    return a, b

# Initialize (a, b) with a log-linear fit on positive flux values
mask_pos = Y > 0
b0, a0 = np.polyfit(H[mask_pos], np.log(Y[mask_pos]), deg=1)

a_poi, b_poi = fit_poisson_mle(H, N, T, a0=a0, b0=b0)

A_poi = np.exp(a_poi)
Y_hat = A_poi * np.exp(b_poi * H)

# Fit metrics
rmse = np.sqrt(np.mean((Y - Y_hat)**2))
SSE = np.sum((Y - Y_hat)**2)
SST = np.sum((Y - Y.mean())**2)
R2  = 1.0 - SSE / SST if SST > 0 else np.nan

# Plot: muon flux vs altitude + fitted curve
fig, ax = plt.subplots(figsize=(10, 6), dpi=130)

ax.scatter(H, Y, s=100, label="Cities")

for x, y, lab in zip(H, Y, L):
    ax.annotate(lab, (x, y), textcoords="offset points", xytext=(6, -6), fontsize=12)

h_grid = np.linspace(H.min(), H.max(), 400)
y_grid = A_poi * np.exp(b_poi * h_grid)

ax.plot(
    h_grid,
    y_grid,
    linewidth=2,
    label=rf"Poisson: $\y(h)={A_poi:.3g}e^{{{b_poi:.8g}·h}}$"
)

metrics_txt = (
    rf"RMSE = {rmse:.3f}" "\n"
    rf"$R^2$ = {R2:.3f}"
)

ax.text(
    0.03, 0.82, metrics_txt,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=12,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray")
)

ax.set_xlabel("Altitude (m)")
ax.set_ylabel(r"part/(m$^2$·s)")
ax.set_title("Muon flux vs altitude")
ax.grid(True, linestyle="--", linewidth=0.6)

ax.legend(loc="upper left", frameon=True)

fig.tight_layout()
fig.savefig(OUT_FLUX_FIG, bbox_inches="tight")
plt.show()